# ДЗ 1.1. Математический маятник

## Часть 1 (зелёная): линейная модель

Уравнение линейного маятника:

$$\ddot\theta + \omega^2 \theta = 0$$

Сводим к системе первого порядка, обозначив $u = \theta$, $v = \dot\theta$:

$$\begin{cases} \dot u = v \\ \dot v = -\omega^2 u \end{cases}$$

Точное решение при начальных условиях $u(0)=u_0$, $v(0)=v_0$:

$$u(t) = u_0 \cos(\omega t) + \frac{v_0}{\omega}\sin(\omega t)$$
$$v(t) = -u_0 \omega \sin(\omega t) + v_0 \cos(\omega t)$$

Будем интегрировать систему методами **явного Эйлера** и **RK4**, сравнивать с точным решением и смотреть, как ведёт себя энергия:

$$T = \frac12 v^2, \quad U = \frac12 \omega^2 u^2, \quad E = T + U$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Параметры модели
omega = 1.0          # собственная частота
u0, v0 = 1.0, 0.0    # начальные условия: отклонение и скорость
s0 = np.array([u0, v0], dtype=np.float64)

# Правая часть системы

In [ ]:
def rhs_linear(state):
    """Правая часть линейного маятника: u' = v, v' = -omega^2 u."""
    u, v = state
    return np.array([v, -omega**2 * u])

# Методы

In [ ]:
def step_euler(state, dt):
    """Явный метод Эйлера первого порядка."""
    return state + dt * rhs_linear(state)


def step_rk4(state, dt):
    """Классический RK4."""
    k1 = rhs_linear(state)
    k2 = rhs_linear(state + dt * k1 / 2)
    k3 = rhs_linear(state + dt * k2 / 2)
    k4 = rhs_linear(state + dt * k3)
    return state + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

# Функция симуляции

In [ ]:
def simulate(step_func, s0, dt, t_end):
    """Прогоняет интегратор и возвращает массивы времени и состояний (N+1, 2)."""
    n_steps = int(round(t_end / dt))
    t = np.linspace(0.0, t_end, n_steps + 1)

    states = np.zeros((n_steps + 1, 2), dtype=np.float64)
    states[0] = s0

    for i in range(1, n_steps + 1):
        states[i] = step_func(states[i - 1], dt)

    return t, states

### Точное решение

Для линейной модели решение известно в явном виде, поэтому можем честно измерить ошибку численного метода.

In [ ]:
def exact_solution(t, u0, v0, omega):
    """Аналитическое решение линейного маятника."""
    u = u0 * np.cos(omega * t) + (v0 / omega) * np.sin(omega * t)
    v = -u0 * omega * np.sin(omega * t) + v0 * np.cos(omega * t)
    return u, v


def kinetic_energy(v):
    return 0.5 * v**2


def potential_energy(u, omega):
    return 0.5 * omega**2 * u**2

### Прогон для нескольких шагов

Возьмём три шага: `dt = 0.1, 0.01, 0.001`. Для каждого посчитаем траекторию обоими методами и точное решение. Так увидим, как ошибка зависит от шага.

In [ ]:
dt_list = [0.1, 0.01, 0.001]
t_end = 30.0

results = {}

for dt in dt_list:
    for name, step_func in [("Euler", step_euler), ("RK4", step_rk4)]:
        t, states = simulate(step_func, s0, dt, t_end)
        u_ex, v_ex = exact_solution(t, u0, v0, omega)
        exact_states = np.column_stack([u_ex, v_ex])
        results[(name, dt)] = (t, states, exact_states)

print("Ключи:", list(results.keys()))

### Ошибка и энергия

Сравним численное решение с точным по норме разницы состояний и посмотрим на поведение полной энергии. У явного Эйлера энергия должна **расти** (метод неустойчив по энергии), у RK4 — колебаться вблизи начального значения с малой амплитудой.

In [ ]:
for dt in dt_list:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for name, color in [("Euler", "tab:red"), ("RK4", "tab:blue")]:
        t, states, exact_states = results[(name, dt)]
        u, v = states[:, 0], states[:, 1]

        # Ошибка по состоянию
        err = np.linalg.norm(states - exact_states, axis=1)

        # Энергии
        T = kinetic_energy(v)
        U = potential_energy(u, omega)
        E = T + U

        axes[0].plot(t, err, color=color, label=name)
        axes[1].plot(t, E, color=color, label=name)
        axes[2].plot(u, v, color=color, label=name)

    # Точная фазовая траектория (на третьей панели)
    t, _, exact_states = results[("Euler", dt)]
    axes[2].plot(exact_states[:, 0], exact_states[:, 1],
                 'k--', linewidth=1, label="точное")

    axes[0].set(xlabel="t", ylabel="||s - s_exact||", title=f"Ошибка, dt={dt}")
    axes[0].set_yscale("log")
    axes[1].set(xlabel="t", ylabel="E", title=f"Полная энергия, dt={dt}")
    axes[2].set(xlabel=r"$\theta$", ylabel=r"$\dot\theta$",
                title=f"Фазовая траектория, dt={dt}")

    for ax in axes:
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Численный анализ: считаем ошибку в фиксированный момент t = t_end
# и оцениваем порядок сходимости p по формуле p ≈ log(err(dt1)/err(dt2)) / log(dt1/dt2)

print("=" * 60)
print("Сравнение Методов: ошибка в момент t = t_end")
print("=" * 60)

t_end = 30.0

# Соберём ошибку в конечный момент для каждого метода и шага
err_final = {name: {} for name in ("Euler", "RK4")}
E_drift   = {name: {} for name in ("Euler", "RK4")}

for dt in dt_list:
    for name in ("Euler", "RK4"):
        t, states, exact_states = results[(name, dt)]

        # Ошибка по состоянию в конечный момент
        err_final[name][dt] = np.linalg.norm(states[-1] - exact_states[-1])

        # Энергия: начальная, конечная и относительный дрейф
        u, v = states[:, 0], states[:, 1]
        E = kinetic_energy(v) + potential_energy(u, omega)
        E0 = E[0]
        E_drift[name][dt] = (E[-1] - E0) / E0

# Таблица ошибок
print(f"\n{'метод':<8}{'dt':>8}{'||s-s_exact|| в t_end':>25}{'отн. дрейф E':>18}")
print("-" * 60)
for name in ("Euler", "RK4"):
    for dt in dt_list:
        print(f"{name:<8}{dt:>8.3f}{err_final[name][dt]:>25.3e}{E_drift[name][dt]:>18.3e}")

# Оценка порядка сходимости
print("\nОценка порядка сходимости p (по уменьшению dt в 10 раз):")
print("-" * 60)
for name in ("Euler", "RK4"):
    dts = sorted(err_final[name].keys(), reverse=True)
    for i in range(len(dts) - 1):
        dt1, dt2 = dts[i], dts[i + 1]
        e1, e2 = err_final[name][dt1], err_final[name][dt2]
        p = np.log(e1 / e2) / np.log(dt1 / dt2)
        print(f"{name:<8} dt {dt1:.3f} -> {dt2:.3f}:  p ≈ {p:.2f}")

print("\nТеория: у явного Эйлера p = 1, у RK4 p = 4.")

### Графики кинетической и потенциальной энергии

Отдельно посмотрим на компоненты механической энергии для линейного маятника:

$$T = \frac12 \dot\theta^2, \qquad U = \frac12 \omega^2 \theta^2, \qquad E = T + U.$$

Для линейной модели энергия сохраняется точно: $E(t) = E(0) = \frac12 \omega^2 \theta_0^2$ при нулевой начальной скорости. В численном решении полная энергия будет слегка колебаться (RK4) или систематически дрейфовать (Эйлер), а вот $T$ и $U$ перетекают друг в друга с частотой $2\omega$ — когда маятник проходит через положение равновесия, $T$ максимальна, а $U$ равна нулю, и наоборот.

In [ ]:
dt_list_energy = [0.1, 0.01, 0.001]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for ax, dt in zip(axes, dt_list_energy):
    t, states, _ = results[("RK4", dt)]
    u, v = states[:, 0], states[:, 1]

    T = kinetic_energy(v)
    U = potential_energy(u, omega)
    E = T + U

    ax.plot(t, T, label="Кинетическая $T$", color="tab:blue")
    ax.plot(t, U, label="Потенциальная $U$", color="tab:orange")
    ax.plot(t, E, label="Полная $E$", linewidth=2, color="black")

    ax.set(xlabel="t", ylabel="Энергия",
           title=f"Энергии, RK4, dt = {dt}")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

### Анимация маятника

Проверим, что траектория выглядит как настоящее колебание. Для анимации возьмём RK4 с малым шагом.

In [ ]:
def animate_pendulum(theta_values, dt, stride=5):
    """Анимация маятника"""
    length = 1.0
    x = length * np.sin(theta_values)
    y = -length * np.cos(theta_values)

    frames = range(0, len(theta_values), stride)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(-1.2 * length, 1.2 * length)
    ax.set_ylim(-1.2 * length, 1.2 * length)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.axhline(0, color="gray", linewidth=1)

    rod, = ax.plot([], [], "k-", linewidth=1.5)
    bob, = ax.plot([], [], "o", color="crimson", markersize=9)
    pivot, = ax.plot([0], [0], "ko", markersize=5)

    def update(i):
        rod.set_data([0, x[i]], [0, y[i]])
        bob.set_data([x[i]], [y[i]])
        return rod, bob

    anim = animation.FuncAnimation(
        fig, update, frames=frames, interval=20, blit=True
    )
    plt.close(fig)
    return anim

In [ ]:
# Берём траекторию RK4
t, states, _ = results[("RK4", 0.01)]
theta_vals = states[:, 0]

anim = animate_pendulum(theta_vals, dt=0.01, stride=5)

In [ ]:
from IPython.display import HTML
HTML(anim.to_jshtml())

## Часть 2 (жёлтая): нелинейный маятник

В линейной модели мы заменяли $\sin\theta \approx \theta$. Это работает только при малых углах. Настоящее уравнение маятника:

$$\ddot\theta + \omega^2 \sin\theta = 0$$

В виде системы:

$$\begin{cases} \dot u = v \\ \dot v = -\omega^2 \sin u \end{cases}$$

Точного решения в элементарных функциях нет (оно выражается через эллиптические функции), поэтому будем считать численно и сравнивать с линейной моделью.

Главный физический эффект: **период нелинейного маятника зависит от амплитуды**, тогда как у линейного он равен $2\pi/\omega$ и от амплитуды не зависит.

# Часть A. Нелинейная модель
## Нелинейная правая часть

In [ ]:
def rhs_nonlinear(state):
    """Правая часть нелинейного маятника: u' = v, v' = -omega^2 sin(u)."""
    u, v = state
    return np.array([v, -omega**2 * np.sin(u)])


def step_euler_nonlinear(state, dt):
    return state + dt * rhs_nonlinear(state)


def step_rk4_nonlinear(state, dt):
    k1 = rhs_nonlinear(state)
    k2 = rhs_nonlinear(state + dt * k1 / 2)
    k3 = rhs_nonlinear(state + dt * k2 / 2)
    k4 = rhs_nonlinear(state + dt * k3)
    return state + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

### Как измерять период численно

Период — это время между двумя последовательными прохождениями положения равновесия $\theta = 0$ в одном и том же направлении (например, оба раза слева направо). Проще всего:

1. Найти все моменты $t_i$, где $u_i \cdot u_{i+1} < 0$ — это смены знака, то есть прохождения через ноль.
2. Между двумя такими моментами (через один) и есть период.

Чтобы получить период точнее, чем шаг сетки, сделаем **линейную интерполяцию** момента перехода через ноль: если $u_i > 0$ и $u_{i+1} < 0$, то ноль где-то между $t_i$ и $t_{i+1}$, и его положение оценивается как

$$t^* = t_i + dt \cdot \frac{u_i}{u_i - u_{i+1}}.$$

# Функция поиска периода

In [ ]:
def measure_period(t, u):
    """Возвращает средний период по нескольким прохождениям нуля (слева направо)."""
    # Индексы, где знак u меняется с + на - (переход через ноль в одну сторону)
    crossings = []
    for i in range(len(u) - 1):
        if u[i] > 0 and u[i + 1] <= 0:
            # линейная интерполяция момента перехода
            t_star = t[i] + (t[i+1] - t[i]) * u[i] / (u[i] - u[i+1])
            crossings.append(t_star)

    if len(crossings) < 2:
        return None

    # Берём разности между соседними переходами — это отдельные периоды
    periods = np.diff(crossings)
    return np.mean(periods), periods

### Зависимость периода от амплитуды

Запустим нелинейный маятник с разными начальными углами $\theta_0$ (скорость $v_0 = 0$) и измерим период. Сравним с:

- **линейной теорией** $T_\text{лин} = 2\pi/\omega$;
- **точным выражением через эллиптический интеграл**:

$$T(\theta_0) = \frac{4}{\omega} K\!\left(\sin\frac{\theta_0}{2}\right),$$

где $K(k)$ — полный эллиптический интеграл первого рода.

In [ ]:
from scipy.special import ellipk

# Амплитуды от 1° до почти 180°
amplitudes_deg = np.array([1, 2, 5, 10, 15, 20, 30, 45, 60, 90, 120, 150, 170])
amplitudes_rad = np.deg2rad(amplitudes_deg)

dt_period = 0.001
t_end_period = 60.0

periods_numeric = []
periods_exact_formula = []

for a in amplitudes_rad:
    s_init = np.array([a, 0.0])
    t, states = simulate(step_rk4_nonlinear, s_init, dt_period, t_end_period)
    u = states[:, 0]

    res = measure_period(t, u)
    periods_numeric.append(res[0] if res is not None else np.nan)

    # Точная формула
    k = np.sin(a / 2)
    periods_exact_formula.append(4.0 / omega * ellipk(k**2))

periods_numeric = np.array(periods_numeric)
periods_exact_formula = np.array(periods_exact_formula)
T_linear = 2 * np.pi / omega

print(f"{'θ0, град':>10}{'T числ.':>12}{'T формула':>12}{'T лин.':>10}{'отн. откл. от лин.':>22}")
print("-" * 66)
for adeg, tn, tf in zip(amplitudes_deg, periods_numeric, periods_exact_formula):
    rel = (tn - T_linear) / T_linear
    print(f"{adeg:>10}{tn:>12.6f}{tf:>12.6f}{T_linear:>10.4f}{rel:>22.4%}")

### График: период от амплитуды

Построим:
1. Кривую $T(\theta_0)$ из численного счёта.
2. Точную формулу (маркеры).
3. Горизонтальную линию $T_\text{лин} = 2\pi/\omega$.

И отдельно — **относительное отклонение** $(T - T_\text{лин}) / T_\text{лин}$ в процентах.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(amplitudes_deg, periods_numeric, "o-", label="численно (RK4)")
axes[0].plot(amplitudes_deg, periods_exact_formula, "x", color="black",
             markersize=8, label="формула через K(k)")
axes[0].axhline(T_linear, color="gray", linestyle="--", label=r"$T_{лин} = 2\pi/\omega$")
axes[0].set(xlabel=r"$\theta_0$, градусы", ylabel="Период T",
            title="Зависимость периода от амплитуды")
axes[0].legend()
axes[0].grid(alpha=0.3)

rel_dev = (periods_numeric - T_linear) / T_linear * 100
axes[1].plot(amplitudes_deg, rel_dev, "o-", color="crimson")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(xlabel=r"$\theta_0$, градусы",
            ylabel="(T − T_лин)/T_лин, %",
            title="Относительное отклонение от линейной модели")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Найдём, при какой амплитуде отклонение превышает 1% и 5%
print("Когда линейная модель ещё достаточно точна для предсказания периода?")
for threshold in (1.0, 5.0):
    idx = np.argmax(np.abs(rel_dev) > threshold)
    if np.abs(rel_dev[idx]) > threshold:
        print(f"Отклонение > {threshold}% начиная с θ0 ≈ {amplitudes_deg[idx]}°")

### Когда линейная модель ещё достаточно точна для предсказания периода?

По результатам численного анализа зависимости периода нелинейного маятника
от начальной амплитуды $\theta_0$:

> При $\theta_0 \approx 30°$ отклонение периода превышает **1%**
>
> При $\theta_0 \approx 60°$ отклонение периода превышает **5%**

**Вывод:** линейная модель $\ddot\theta + \omega^2\theta = 0$ даёт корректный
прогноз периода при погрешности:

- **до 1%** — для амплитуд $\theta_0 \lesssim 30°$
- **до 5%** — для амплитуд $\theta_0 \lesssim 60°$

> **Причина:** при разложении $\sin\theta = \theta - \theta^3/6 + \ldots$
> отбрасываемый кубический член становится заметным именно при углах
> порядка 20°–60°: сначала он даёт ~1% ошибки в периоде, а при
> $\theta_0 \to 60°$ — уже около 5%.

# Часть B. Расхождение с линейной моделью + сходимость по шагу

### Расхождение нелинейной и линейной моделей для выбранной амплитуды

Возьмём θ0 = 60° — там нелинейность уже хорошо видна. Запустим линейную и нелинейную модели с одинаковыми начальными условиями и одним методом (RK4, мелкий шаг), и посмотрим, как расходятся траектории во времени.

In [ ]:
theta0_deg = 60
theta0 = np.deg2rad(theta0_deg)
s_init = np.array([theta0, 0.0])

dt_cmp = 0.001
t_end_cmp = 20.0

# Нелинейная модель
t, states_nl = simulate(step_rk4_nonlinear, s_init, dt_cmp, t_end_cmp)
u_nl = states_nl[:, 0]

# Линейная модель с теми же начальными условиями
t, states_lin = simulate(step_rk4, s_init, dt_cmp, t_end_cmp)
u_lin = states_lin[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(t, u_nl, label="нелинейная")
axes[0].plot(t, u_lin, "--", label="линейная")
axes[0].set(xlabel="t", ylabel=r"$\theta$", title=f"θ0 = {theta0_deg}°")
axes[0].legend()
axes[0].grid(alpha=0.3)

diff = u_nl - u_lin
axes[1].plot(t, diff, color="crimson")
axes[1].set(xlabel="t", ylabel=r"$\theta_{нелин} - \theta_{лин}$",
            title="Расхождение моделей")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Сходимость по шагу для нелинейной модели

Проверим, что RK4 и для нелинейной задачи даёт 4-й порядок точности. Считать точное решение мы не можем (его нет), поэтому возьмём в качестве «эталона» решение при очень мелком шаге (dt = 1e-5) и будем сравнивать с ним.

In [ ]:
theta0 = np.deg2rad(60)
s_init = np.array([theta0, 0.0])
t_end_ref = 10.0

# Эталонное решение
dt_ref = 1e-5
t_ref, states_ref = simulate(step_rk4_nonlinear, s_init, dt_ref, t_end_ref)
s_ref_final = states_ref[-1]

# Проверяемые шаги
dts = [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
errs = []

for dt in dts:
    t, states = simulate(step_rk4_nonlinear, s_init, dt, t_end_ref)
    err = np.linalg.norm(states[-1] - s_ref_final)
    errs.append(err)

errs = np.array(errs)

print(f"{'dt':>8}{'ошибка':>15}{'p (оценка)':>15}")
print("-" * 40)
print(f"{dts[0]:>8.4f}{errs[0]:>15.3e}{'-':>15}")
for i in range(1, len(dts)):
    p = np.log(errs[i-1] / errs[i]) / np.log(dts[i-1] / dts[i])
    print(f"{dts[i]:>8.4f}{errs[i]:>15.3e}{p:>15.2f}")

# График сходимости в лог-лог
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(dts, errs, "o-", label="RK4 (нелин.)")

# Референсные прямые для порядка 4
ref = errs[-1] * (np.array(dts) / dts[-1])**4
ax.loglog(dts, ref, "--", color="gray", label=r"наклон 4 ($\propto dt^4$)")

ax.set(xlabel="dt", ylabel="Ошибка в t=10")
ax.set_title("Сходимость RK4 для нелинейного маятника")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

# Часть C. Анимации

### Анимация 1: маятник и фазовый след (линейная vs нелинейная модель)

Две панели:
- **слева** — сам маятник (положение стержня и груза);
- **справа** — фазовая траектория: точка, бегущая по фазовой плоскости, оставляя след.

Накладываем линейную и нелинейную модели при θ0 = 60° — видно, что фазовая траектория нелинейного маятника не эллипс, а «искажённый» овал, и что модели постепенно расходятся.

In [ ]:
# Подготовим данные
theta0_deg = 60
theta0 = np.deg2rad(theta0_deg)
s_init = np.array([theta0, 0.0])
dt_anim = 0.01
t_end_anim = 20.0

t, states_nl = simulate(step_rk4_nonlinear, s_init, dt_anim, t_end_anim)
_, states_lin = simulate(step_rk4, s_init, dt_anim, t_end_anim)

u_nl, v_nl = states_nl[:, 0], states_nl[:, 1]
u_lin, v_lin = states_lin[:, 0], states_lin[:, 1]

L = 1.0
x_nl = L * np.sin(u_nl)
y_nl = -L * np.cos(u_nl)
x_lin = L * np.sin(u_lin)
y_lin = -L * np.cos(u_lin)

# Прореживание кадров, чтобы анимация не тормозила
stride = 5
frames = range(0, len(t), stride)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

# ==== Левая панель: маятники ====
ax1 = axes[0]
ax1.set_xlim(-1.2*L, 1.2*L)
ax1.set_ylim(-1.2*L, 1.2*L)
ax1.set_aspect("equal")
ax1.grid(alpha=0.3)
ax1.axhline(0, color="gray", linewidth=1)
ax1.set_title(f"Маятник, θ0 = {theta0_deg}°")

rod_nl,  = ax1.plot([], [], "-",  color="tab:blue",   linewidth=1.5, label="нелин.")
rod_lin, = ax1.plot([], [], "--", color="tab:orange", linewidth=1.5, label="лин.")
bob_nl,  = ax1.plot([], [], "o",  color="tab:blue",   markersize=9)
bob_lin, = ax1.plot([], [], "o",  color="tab:orange", markersize=7, alpha=0.8)
ax1.plot([0], [0], "ko", markersize=5)
ax1.legend(loc="upper right", fontsize=9)

# ==== Правая панель: фазовый след ====
ax2 = axes[1]
ax2.set_xlim(-theta0*1.2, theta0*1.2)
ax2.set_ylim(-theta0*1.2, theta0*1.2)
ax2.set_aspect("equal")
ax2.grid(alpha=0.3)
ax2.set_xlabel(r"$\theta$")
ax2.set_ylabel(r"$\dot\theta$")
ax2.set_title("Фазовый след")

trace_nl,  = ax2.plot([], [], "-", color="tab:blue",   linewidth=1)
trace_lin, = ax2.plot([], [], "-", color="tab:orange", linewidth=1)
dot_nl,    = ax2.plot([], [], "o", color="tab:blue",   markersize=6)
dot_lin,   = ax2.plot([], [], "o", color="tab:orange", markersize=5, alpha=0.8)

def update(i):
    # Маятники
    rod_nl.set_data([0, x_nl[i]], [0, y_nl[i]])
    bob_nl.set_data([x_nl[i]], [y_nl[i]])
    rod_lin.set_data([0, x_lin[i]], [0, y_lin[i]])
    bob_lin.set_data([x_lin[i]], [y_lin[i]])

    # Фазовый след
    trace_nl.set_data(u_nl[:i+1], v_nl[:i+1])
    trace_lin.set_data(u_lin[:i+1], v_lin[:i+1])
    dot_nl.set_data([u_nl[i]], [v_nl[i]])
    dot_lin.set_data([u_lin[i]], [v_lin[i]])

    return rod_nl, bob_nl, rod_lin, bob_lin, trace_nl, trace_lin, dot_nl, dot_lin

anim1 = animation.FuncAnimation(
    fig, update, frames=frames, interval=20, blit=True
)

plt.close(fig)
from IPython.display import HTML
HTML(anim1.to_jshtml())

### Анимация 2: ансамбль маятников разной длины

Возьмём 10 маятников разной длины. Для математического маятника
$\omega = \sqrt{g/L}$, то есть частота зависит от длины. Выберем частоты
**гармонически соизмеримыми** — кратными $\omega_0 = 0.2$:

$$\omega_i = 0.2 \cdot (5 + i), \quad i = 0, 1, \dots, 9
\quad \Longrightarrow \quad \omega = 1.0,\ 1.2,\ \dots,\ 2.8.$$

Все маятники запускаем с **одинаковым начальным углом** $\theta_0 = 30°$
и нулевой скоростью.

In [ ]:
N_pend = 10
g = 1.0
omegas = 0.2 * np.arange(5, 5 + N_pend)      # 1.0, 1.2, ..., 2.8
lengths = g / omegas**2

theta0_ens = np.deg2rad(30)

dt_ens = 0.01
t_end_ens = 20.0

ens_trajectories = []
for w in omegas:
    def rhs_local(state, w=w):
        u, v = state
        return np.array([v, -w**2 * np.sin(u)])

    def step_local(state, dt, w=w):
        k1 = rhs_local(state)
        k2 = rhs_local(state + dt * k1 / 2)
        k3 = rhs_local(state + dt * k2 / 2)
        k4 = rhs_local(state + dt * k3)
        return state + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

    s_init = np.array([theta0_ens, 0.0])
    t, states = simulate(step_local, s_init, dt_ens, t_end_ens)
    ens_trajectories.append(states[:, 0])

ens_trajectories = np.array(ens_trajectories)
t_ens = np.linspace(0, t_end_ens, ens_trajectories.shape[1])

stride = 10
interval = 80                    # мс между кадрами

frames = range(0, len(t_ens), stride)

colors = plt.cm.viridis(np.linspace(0.1, 0.85, N_pend))
#colors = plt.cm.coolwarm(np.linspace(0.05, 0.95, N_pend))

L_max = lengths.max()

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(-1.1*L_max, 1.1*L_max)
ax.set_ylim(-1.1*L_max, 0.3*L_max)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.axhline(0, color="gray", linewidth=1, alpha=0.4)
ax.plot([0], [0], "ko", markersize=6)
ax.set_title(f"Ансамбль из {N_pend} маятников разной длины\n"
             f"(θ0 = 30°, ω = 1.0..2.8)")

rods = [ax.plot([], [], "-", color=colors[i], linewidth=1.8)[0]
        for i in range(N_pend)]
bobs = [ax.plot([], [], "o", color=colors[i], markersize=8)[0]
        for i in range(N_pend)]

def update_ens(i):
    for k in range(N_pend):
        theta = ens_trajectories[k, i]
        L = lengths[k]
        x = L * np.sin(theta)
        y = -L * np.cos(theta)
        rods[k].set_data([0, x], [0, y])
        bobs[k].set_data([x], [y])
    return rods + bobs

anim2 = animation.FuncAnimation(
    fig, update_ens, frames=frames, interval=interval, blit=True
)
plt.close(fig)

HTML(anim2.to_jshtml(fps=10))

### Анимация 3: ансамбль маятников одинаковой длины, разные θ0

Теперь все маятники одинаковой длины (одинаковая $\omega$), но запускаются с разными начальными углами. Период у них будет **разным** — как раз из-за зависимости $T(\theta_0)$. Хорошо видно, как со временем маятники «рассинхронизируются», хотя стартовали одновременно.

In [ ]:
N_pend = 10
omega_fixed = 1.0
theta0_list = np.deg2rad(np.linspace(5, 80, N_pend))

dt_ens = 0.005
t_end_ens = 15.0

def rhs_w(state):
    u, v = state
    return np.array([v, -omega_fixed**2 * np.sin(u)])

def step_w(state, dt):
    k1 = rhs_w(state)
    k2 = rhs_w(state + dt * k1 / 2)
    k3 = rhs_w(state + dt * k2 / 2)
    k4 = rhs_w(state + dt * k3)
    return state + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

trajs = []
for th in theta0_list:
    s_init = np.array([th, 0.0])
    t, states = simulate(step_w, s_init, dt_ens, t_end_ens)
    trajs.append(states[:, 0])

trajs = np.array(trajs)
t_arr = np.linspace(0, t_end_ens, trajs.shape[1])

L_ens = 1.0
stride = 5
frames = range(0, len(t_arr), stride)
colors = plt.cm.plasma(np.linspace(0, 1, N_pend))

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.axhline(0, color="gray", linewidth=1)
ax.plot([0], [0], "ko", markersize=6)
ax.set_title(f"Ансамбль из {N_pend} маятников (одинаковая длина, разные θ0)")

rods = [ax.plot([], [], "-", color=colors[i], linewidth=1.5)[0] for i in range(N_pend)]
bobs = [ax.plot([], [], "o", color=colors[i], markersize=7)[0] for i in range(N_pend)]

def update_ens2(i):
    for k in range(N_pend):
        theta = trajs[k, i]
        x = L_ens * np.sin(theta)
        y = -L_ens * np.cos(theta)
        rods[k].set_data([0, x], [0, y])
        bobs[k].set_data([x], [y])
    return rods + bobs

anim3 = animation.FuncAnimation(
    fig, update_ens2, frames=frames, interval=20, blit=True
)
plt.close(fig)
HTML(anim3.to_jshtml())